# Test de personalidad
## 02 · Análisis exploratorio de datos (EDA)

Proyecto de Sistemas Inteligentes · UNAM · Matemáticas Aplicadas y Computación

## Objetivo

Conocer los datos antes de modelar: ver cómo se distribuye cada variable, cómo se relaciona con el perfil (**Introvertido** o **No introvertido**, definido en el cuaderno 01) y qué tan completa y confiable es la información.

El análisis tiene cuatro partes: **calidad de datos y valores faltantes**, **univariado** (una variable a la vez), **bivariado** (cada variable contra el perfil) y **multivariado** (varias variables juntas).

## Datos

Usamos `Datos_Con_Perfil.xlsx`, el archivo que generó el cuaderno 01. Súbelo al panel de archivos de Colab (o ejecuta la celda de carga y elige el archivo). Recuerda que al cerrar la sesión de Colab los archivos subidos se borran.

**Aviso importante:** son muy pocas personas (menos de 25). Los números sirven para **explorar y describir**, no para sacar conclusiones firmes.

### Preparar el entorno
*Qué hace:* importa las herramientas y revisa si estamos en Google Colab.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files  # solo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

print("pandas", pd.__version__, "| ejecutando en Colab:", EN_COLAB)

### Cargar los datos
*Qué hace:* busca `Datos_Con_Perfil.xlsx` (si no está, te pide subirlo) y lo lee.

In [ ]:
ARCHIVO = "Datos_Con_Perfil.xlsx"

if not os.path.exists(ARCHIVO):
    if EN_COLAB:
        print("Elige el archivo Datos_Con_Perfil.xlsx en la ventana que se abre...")
        files.upload()
    if not os.path.exists(ARCHIVO):
        raise FileNotFoundError(
            f"No se encontró '{ARCHIVO}'. Súbelo al panel de archivos de Colab con ese nombre exacto."
        )

df = pd.read_excel(ARCHIVO, sheet_name="DatosConPerfil")
print(f"Personas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(df["perfil"].value_counts())

### Conjunto de análisis y variables
*Qué hace:* nos quedamos con las personas que tienen perfil (dejamos fuera las que están "Por revisar") y elegimos las variables que **no son texto libre**. Las respuestas de texto libre no se pueden usar directamente en los modelos sin convertirlas a categorías con reglas, y eso se hace en un paso aparte.

Las variables que **no** entran: `pers_q06` y `pers_q07` (con ellas se creó el perfil) y `amor_semestre` (es igual para todas las personas).

In [ ]:
df_a = df[df["perfil"].isin(["Introvertido", "No introvertido"])].copy()
print(f"Personas en el análisis: {len(df_a)} (fuera: {len(df) - len(df_a)} 'Por revisar')")
print(df_a["perfil"].value_counts(), "\n")

VARS_NUM = [
    "mus_tiempo_min_horas",        # horas al día escuchando música (mínimo)
    "mus_tiempo_max_horas",        # horas al día escuchando música (máximo)
    "amor_edad_primera_relacion",  # edad de la primera relación
    "amor_dif_edad_max_anios",     # diferencia de edad máxima aceptada con la pareja
]
VARS_CAT = [
    "pers_genero",       # género
    "amor_pareja",       # ¿tiene pareja?
    "mus_tiempo_tipo",   # tipo de respuesta del tiempo de escucha (puntual, intervalo...)
]

for col in VARS_NUM:
    df_a[col] = pd.to_numeric(df_a[col], errors="coerce")
print("Semestre distinto en los datos:", df_a["amor_semestre"].dropna().unique())

## 1. Calidad de datos y valores faltantes

### Edad igual a 0
En la limpieza se encontró que tres personas escribieron **0** como edad de su primera relación. No es una edad real; podría significar "nunca he tenido pareja", pero eso no lo sabemos. En la limpieza se conservó tal cual. **Para el análisis la tratamos como dato faltante** y lo dejamos documentado.
*Qué hace la celda:* cuenta esos casos y los convierte en vacíos solo en `df_e` (una copia para el EDA).

In [ ]:
df_e = df_a.copy()
col_edad = "amor_edad_primera_relacion"
ceros = df_e[col_edad] == 0
print("Personas con edad 0 en el análisis:", int(ceros.sum()))
df_e.loc[ceros, col_edad] = np.nan

### Valores faltantes
*Qué hace:* cuenta cuántos datos faltan en cada variable. Hay dos orígenes distintos: las personas que **no respondieron** el formulario de relaciones amorosas, y los valores que **no se pudieron convertir** a número (respuestas ambiguas o "7 horas o más", que no tiene máximo).

In [ ]:
cols = VARS_NUM + VARS_CAT
nulos = df_e[cols].isna().sum().to_frame("faltantes")
nulos["porcentaje"] = (nulos["faltantes"] / len(df_e) * 100).round(1)
display(nulos)

print("Personas sin formulario de relaciones amorosas:", int((df_e["tiene_amor"] == "No").sum()))

### Imputación (qué haremos con los faltantes)

- **No inventamos datos.** Con tan pocas personas, imputar tiene riesgos: un valor "típico" puede distorsionar mucho.
- **Para explorar:** en las gráficas simplemente no se cuentan los vacíos (cada gráfica indica con cuántas personas se hizo).
- **Para modelar:** se usará la **mediana** en las variables numéricas y la categoría **"Sin dato"** en las categóricas, calculadas **dentro de la validación cruzada** (solo con los datos de entrenamiento), para que el modelo no "vea" información de la persona que está evaluando.
- `amor_dif_edad_max_anios` tiene muy pocos datos: imputarla inventaría la mayor parte de la columna. Se evaluará si conviene dejarla fuera.

*Qué hace la celda:* muestra qué valor tomaría cada variable con ese criterio, solo como referencia.

In [ ]:
referencia = {}
for col in VARS_NUM:
    referencia[col] = df_e[col].median()
print("Mediana de referencia (solo con los datos disponibles):")
display(pd.Series(referencia).round(2).to_frame("mediana"))

## 2. Análisis univariado

### Variables numéricas
*Qué hace:* resume cada variable (cuántas personas, promedio, mínimo, máximo...) y dibuja su histograma.

In [ ]:
display(df_e[VARS_NUM].describe().round(2))

fig, ejes = plt.subplots(2, 2, figsize=(10, 7))
for eje, col in zip(ejes.ravel(), VARS_NUM):
    datos = df_e[col].dropna()
    eje.hist(datos, bins=8)
    eje.set_title(f"{col}\n(n = {len(datos)})", fontsize=10)
    eje.set_ylabel("Personas")
plt.tight_layout()
plt.show()

### Variables categóricas
*Qué hace:* cuenta cuántas personas hay en cada categoría. "Sin dato" indica que no hay información.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for eje, col in zip(ejes, VARS_CAT):
    conteo = df_e[col].fillna("Sin dato").value_counts()
    eje.bar(conteo.index, conteo.values)
    eje.set_title(col, fontsize=10)
    eje.set_ylabel("Personas")
    plt.setp(eje.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

for col in VARS_CAT:
    print(df_e[col].fillna("Sin dato").value_counts().to_dict())

## 3. Análisis bivariado (cada variable contra el perfil)

### Variables numéricas
*Qué hace:* compara cada variable entre **Introvertido** y **No introvertido**: una tabla con cuántas personas, promedio y mediana, y cajas (boxplots) para ver la dispersión.

In [ ]:
resumen = df_e.groupby("perfil")[VARS_NUM].agg(["count", "mean", "median"]).round(2)
display(resumen.T)

PERFILES = ["Introvertido", "No introvertido"]
fig, ejes = plt.subplots(1, 4, figsize=(15, 4))
for eje, col in zip(ejes, VARS_NUM):
    grupos = [df_e.loc[df_e["perfil"] == p, col].dropna() for p in PERFILES]
    eje.boxplot(grupos)
    eje.set_xticks([1, 2])
    eje.set_xticklabels(["Introv.", "No introv."])
    eje.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

### Variables categóricas
*Qué hace:* cruza cada variable con el perfil. La primera tabla cuenta personas; la segunda muestra el porcentaje de cada perfil dentro de cada categoría.

In [ ]:
for col in VARS_CAT:
    cruce = pd.crosstab(df_e[col].fillna("Sin dato"), df_e["perfil"])
    print(f"--- {col} (personas) ---")
    display(cruce)
    print(f"--- {col} (% de cada fila) ---")
    display((pd.crosstab(df_e[col].fillna("Sin dato"), df_e["perfil"], normalize="index") * 100).round(0))

## 4. Análisis multivariado (varias variables a la vez)

### Correlaciones
*Qué hace:* mide qué tan relacionadas están las variables numéricas entre sí (correlación de Spearman, que no se afecta tanto por valores extremos). También cuenta cuántas personas hay por cada par: **con pocas personas, una correlación puede ser pura casualidad**.

In [ ]:
corr = df_e[VARS_NUM].corr(method="spearman")
display(corr.round(2))

disponible = df_e[VARS_NUM].notna().astype(int)
print("Personas con datos en ambas variables de cada par:")
display(disponible.T.dot(disponible))

fig, eje = plt.subplots(figsize=(6, 5))
imagen = eje.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm")
eje.set_xticks(range(len(VARS_NUM)))
eje.set_yticks(range(len(VARS_NUM)))
eje.set_xticklabels(VARS_NUM, rotation=45, ha="right", fontsize=8)
eje.set_yticklabels(VARS_NUM, fontsize=8)
for i in range(len(VARS_NUM)):
    for j in range(len(VARS_NUM)):
        valor = corr.values[i, j]
        if not np.isnan(valor):
            eje.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(imagen)
plt.title("Correlación de Spearman")
plt.tight_layout()
plt.show()

### Variables numéricas coloreadas por perfil
*Qué hace:* dibuja cada variable contra las demás, con un color por perfil, para ver si alguna combinación separa a los dos grupos.

In [ ]:
COLORES = {"Introvertido": "tab:blue", "No introvertido": "tab:orange"}
n = len(VARS_NUM)
fig, ejes = plt.subplots(n, n, figsize=(10, 10))
for i, fila in enumerate(VARS_NUM):
    for j, columna in enumerate(VARS_NUM):
        eje = ejes[i, j]
        if i == j:
            eje.hist(df_e[fila].dropna(), bins=8, color="gray")
        else:
            for perfil, color in COLORES.items():
                sub = df_e[df_e["perfil"] == perfil]
                eje.scatter(sub[columna], sub[fila], color=color, s=20)
        if i == n - 1:
            eje.set_xlabel(columna, fontsize=7)
        if j == 0:
            eje.set_ylabel(fila, fontsize=7)
        eje.tick_params(labelsize=7)
fig.suptitle("Azul = Introvertido · Naranja = No introvertido", fontsize=10)
plt.tight_layout()
plt.show()

### Perfil, pareja y música juntos
*Qué hace:* mira el tiempo de escucha de música según el perfil **y** si la persona tiene pareja.

In [ ]:
combinado = (
    df_e.assign(amor_pareja=df_e["amor_pareja"].fillna("Sin dato"))
    .groupby(["perfil", "amor_pareja"])["mus_tiempo_min_horas"]
    .agg(["count", "mean"])
    .round(2)
)
display(combinado)

## Conclusiones

**Datos y calidad**
- Analizamos **21 personas**: 14 Introvertidas y 7 No introvertidas (las 2 "Por revisar" quedan fuera).
- Faltan muchos datos en las variables de relaciones amorosas: la edad de la primera relación tiene **33 % de vacíos** (4 personas no respondieron ese formulario y 3 escribieron 0) y la diferencia de edad máxima tiene **57 %**. Las variables de música están casi completas.
- Hay un valor extremo de **15 horas** de escucha al día, que se dejó sin modificar.

**Lo que vemos al comparar con el perfil (solo descriptivo)**
- **Género:** 5 de las 6 mujeres son Introvertidas (83 %), frente a 9 de 15 hombres (60 %). Son muy pocas mujeres para sacar una conclusión.
- **Pareja:** casi no cambia el perfil entre quienes tienen pareja y quienes no (alrededor de 6 de cada 10 son Introvertidas en ambos casos).
- **Tiempo de escucha:** la mediana es 3 horas en Introvertidos y 4 en No introvertidos. La diferencia es pequeña.
- **Edad de la primera relación:** promedio de 15.8 años en Introvertidos y 14 en No introvertidos, con solo 9 y 5 personas.

**Cuidados para el modelado**
- **Ninguna de estas variables separa claramente a los dos perfiles.** Con lo que tenemos, un modelo tendrá muy poca información para aprender.
- `mus_tiempo_min_horas` y `mus_tiempo_max_horas` casi son la misma variable (correlación 0.99): usar solo una.
- `mus_tiempo_tipo` **no debe usarse como predictora.** Todas las personas que escribieron un intervalo, un mínimo o una respuesta ambigua están en el grupo Introvertido, pero eso refleja cómo escribieron la respuesta, no su personalidad.
- Las correlaciones con `amor_dif_edad_max_anios` se calculan con 9 personas o menos: no son confiables.
- Los resultados sirven para explorar, no para afirmar relaciones.

**Siguiente paso:** como las variables numéricas y categóricas disponibles aportan poco, el siguiente paso es **construir variables nuevas a partir de las respuestas de texto**, con reglas claras (por ejemplo, el rol que cada persona toma en un equipo o cómo reacciona ante un cambio de planes), y después pasar al modelado.